In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
                faiss-cpu python-dotenv

In [40]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")



In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Everything imported successfully!")

/tmp/ipykernel_7436/63873068.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Everything imported successfully!


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

print("Everything imported successfully!")

Everything imported successfully!


In [5]:
video_id= "Gfr50f6ZBvo"
transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
transcript= " ".join(chunk.text for chunk  in transcript_list)


Text Splitter

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)

In [14]:
chunks=splitter.create_documents([transcript])

In [15]:
len(chunks)

48

Vector Store creation


In [16]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key,
    batch_size=5
)

In [17]:
vector_store=FAISS.from_documents(chunks, embeddings)

In [18]:
test_embedding = embeddings.embed_query("Hello, this is a test.")

print(len(test_embedding))

3072


In [20]:
vector_store.index_to_docstore_id

{0: '05ced541-a99a-4d9d-8e7f-80f3cc097481',
 1: 'cafdfafe-67fe-4d37-9bbf-08f3a7159fca',
 2: '53d93574-a76c-4036-8379-a9845cb3adc8',
 3: '5f65ac28-f225-4cf4-9ec6-0ea20ca1280f',
 4: 'ce683393-4f3d-4088-8997-39596294b095',
 5: '7f31523f-4f70-417e-a785-f098e85ecb21',
 6: 'e0fb2097-729a-46bc-b962-46c59a2af8df',
 7: '62b89c56-fd2a-41d6-b28b-b6267bd8e53e',
 8: '38122076-e10f-41f7-9690-d0140545ccdf',
 9: '12f9822a-212e-4dd7-9a1f-f67c1e2649ee',
 10: 'e3231a02-c97c-48f7-941f-8dca9ac6afa4',
 11: '6c1a6ec8-983b-4d3e-942f-dcf4f324dbea',
 12: 'd57b88f9-5fff-48cf-9c02-aa115b609aa5',
 13: '2dcf2f9b-ea06-42b6-ab38-88e90d936a04',
 14: '16cb6a3a-3865-4901-a9bc-6b442005c248',
 15: '3972b36d-c43e-426d-84e5-8deab0a901f7',
 16: '952d8d60-52b4-4fde-bc09-9df4cd40eda8',
 17: 'cf8e8e8a-90c5-4ed9-9ec0-6ca989c9d0a2',
 18: '9a23e01f-76fe-4f0a-891b-462d115c7c0b',
 19: 'd7d9e551-e415-4dde-9ecc-595182b232e6',
 20: 'c337e443-1779-4273-9a42-87458a991beb',
 21: '6c86bceb-50c0-40f4-b783-0b165f1a6e59',
 22: 'b7056420-1244-

Retrieval

In [21]:
retriever= vector_store.as_retriever(search_type="mmr", seach_kwargs={"k":4})

In [22]:
retriever.invoke("what is deepmind")

[Document(id='e0fb2097-729a-46bc-b962-46c59a2af8df', metadata={}, page_content="we've used of ai is in deep mind from the beginning which is using games as a testing ground for proving out ai algorithms and developing ai algorithms and that was a that was a sort of um a core component of our vision at the start of deepmind was that we would use games very heavily uh as our main testing ground certainly to begin with um because it's super efficient to use games and also you know it's very easy to have metrics to see how well your systems are improving and what direction your ideas are going in and whether you're making incremental improvements and because those games are often rooted in something that humans did for a long time beforehand there's already a strong set of rules like it's already a damn good benchmark yes it's really good for so many reasons because you've got you've got you've got clear measures of how good humans can be at these things and in some cases like go we've bee

Augmentation


In [23]:
llm= ChatGoogleGenerativeAI(
    model="gemini-3.6-flash")

In [24]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [25]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"

In [26]:
retrieved_docs= retriever.invoke(question)

In [27]:
context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)

In [28]:
final_prompt= prompt.invoke({"context": context_text , "question":question})

In [29]:
answer= llm.invoke(final_prompt)

In [30]:
print(answer.text)

Yes, nuclear fusion is discussed in the transcript. Here is what was discussed:

* **Challenges in Fusion:** Fusion faces major physics, material science, and engineering challenges, particularly in building massive reactors and containing plasma.
* **Collaboration:** They collaborated with EPFL (the Swiss Technical Institute in Switzerland) to use their test reactor for experiments.
* **Bottleneck Identification:** They spoke with fusion experts to identify the bottleneck problems preventing fusion from working and looked for problems that AI methods (specifically reinforcement learning) could address.
* **Plasma Control:** 
  * Plasma reaches temperatures around a million degrees Celsius (hotter than the sun) and cannot be contained by any physical material, requiring powerful superconducting magnetic fields to hold it.
  * Because plasma is highly unstable, controllers need to predict its behavior ahead of time and adjust the magnetic field within milliseconds.
  * Traditional contr

Building a chain

In [31]:
from langchain_core.runnables import  RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [32]:
def format_docs(retrieved_docs):
    context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text
    

In [33]:
parallel_chain= RunnableParallel(
    {
    "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
)

In [34]:
parallel_chain.invoke("who is demis"

{'context': "into the room with the system and have a conversation maybe you only get to ask one question if you do what question would you ask her i would probably ask um what is the true nature of reality i think that's the question i don't know if i'd understand the answer because maybe it would be 42 or something like that but um that's the question i would ask and then there'll be a deep sigh from the systems like all right how do i explain to the excuse me exactly all right let me i don't have time to explain uh maybe i'll draw you a picture that it is i mean how do you even begin um to answer that question well i think it would um what would you what would you think the answer could possibly look like i think it could it could start looking like uh uh more fundamental explanations of physics would be the beginning you know more careful specification of that taking you walking us through by the hand as to what one would do to maybe prove those things out maybe giving you glimpses

In [35]:
parser= StrOutputParser()

In [36]:
chain= parallel_chain | prompt |llm | parser

In [38]:
main_chain=chain.invoke("can you summarise this video")

In [39]:
print(main_chain)

Based on the provided transcript, the conversation covers the following key topics:

* **AlphaFold and Structural Biology:** Demis discusses how computers and AI are helping us understand biology. He highlights AlphaFold 2 as a major breakthrough that solved the "protein folding" problem—predicting how a protein's amino acid sequence folds into a 3D structure, which determines its function and helps in targeting diseases and drugs.
* **Extraterrestrial Life and the Simulation Hypothesis:** The transcript touches on space exploration ideas, including Von Neumann probes and Dyson spheres. Despite searching for cosmic signals, humanity has found silence. Demis discusses arguments around this (such as the "safari view") and compares it to the simulation hypothesis, which suggests reality may have a deeper, computational foundation.
* **Levels of AI Capabilities:** Demis breaks down AI intelligence into three levels:
  1. *Interpolation:* Basic AI generating average examples from training d